# install packages

In [ ]:
install.packages("preventr")

In [ ]:
install.packages("tidyverse")

# load packages

In [ ]:
library(tidyverse)
library(preventr)
library(data.table)

# read in input file

In [ ]:
pheno_pgs = fread('UKBB.CVD.IRM.phenotype.csv', sep = ',')
dim(pheno_pgs)
head(pheno_pgs)

# rename columns, and convert units

In [ ]:
pheno_pgs = pheno_pgs %>%
rename('HbA1c_CAD' = 'HbA1c_%_CAD', 'HbA1c_AFIB' = 'HbA1c_%_AFIB', 'HbA1c_HF' = 'HbA1c_%_HF')

In [ ]:
pheno_pgs = pheno_pgs %>%
mutate(HDL_CAD = HDL_CAD * 38.67,
       HDL_AFIB = HDL_AFIB * 38.67,
       HDL_HF = HDL_HF * 38.67,
       TOTAL_CHOL_CAD = TOTAL_CHOL_CAD * 38.67,
       TOTAL_CHOL_AFIB = TOTAL_CHOL_AFIB * 38.67,
       TOTAL_CHOL_HF = TOTAL_CHOL_HF * 38.67)

# choose eGFR columns

In [ ]:
pheno_pgs %>%
select('eGFR_CREATININE_CAD') %>%
na.omit() %>%
nrow()

In [ ]:
pheno_pgs %>%
select('eGFR_CYS_C_CAD') %>%
na.omit() %>%
nrow()

# make inputs

## CAD

In [ ]:
summary(pheno_pgs$AGE_CAD)
summary(pheno_pgs$HDL_CAD)
summary(pheno_pgs$TOTAL_CHOL_CAD)
summary(pheno_pgs$BMI_CAD)
summary(pheno_pgs$eGFR_CYS_C_CAD)
summary(pheno_pgs$SBP_CAD)
summary(pheno_pgs$uACR_CAD)
summary(pheno_pgs$HbA1c_CAD)

### subset and rename

In [ ]:
cad_prevent_input = pheno_pgs %>%
select(eid,
       CAD,
       AGE_CAD,
       SEX,
       DIABETES_CAD,
       SBP_CAD,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_CAD,
       HDL_CAD,
       SMOKING_status_CAD,
       eGFR_CYS_C_CAD,
       BMI_CAD) %>%
rename('age' = 'AGE_CAD',
       'sex' = 'SEX',
       'sbp' = 'SBP_CAD',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_CAD',
       'hdl_c' = "HDL_CAD",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_CAD',
       'egfr' = 'eGFR_CYS_C_CAD',
       'dm' = 'DIABETES_CAD',
       'bmi' = 'BMI_CAD') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

unique(cad_prevent_input$sex)
unique(cad_prevent_input$smoking)
summary(cad_prevent_input$age)
summary(cad_prevent_input$hdl_c)
summary(cad_prevent_input$total_c)
nrow(cad_prevent_input)
cad_prevent_input %>%
group_by(CAD) %>%
count()
head(cad_prevent_input)

In [ ]:
cad_prevent_input_hba1c_uacr = pheno_pgs %>%
select(eid,
       CAD,
       AGE_CAD,
       SEX,
       DIABETES_CAD,
       SBP_CAD,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_CAD,
       HDL_CAD,
       SMOKING_status_CAD,
       eGFR_CYS_C_CAD,
       BMI_CAD,
       HbA1c_CAD,
       uACR_CAD) %>%
rename('age' = 'AGE_CAD',
       'sex' = 'SEX',
       'sbp' = 'SBP_CAD',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_CAD',
       'hdl_c' = "HDL_CAD",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_CAD',
       'egfr' = 'eGFR_CYS_C_CAD',
       'dm' = 'DIABETES_CAD',
       'bmi' = 'BMI_CAD',
       'hba1c' = 'HbA1c_CAD',
       'uacr' = 'uACR_CAD') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       uacr = ifelse(uacr < 0.1 | uacr > 25000, NA, uacr),
       hba1c = ifelse(hba1c < 4.5 | hba1c > 15, NA, hba1c),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(cad_prevent_input_hba1c_uacr)
cad_prevent_input_hba1c_uacr %>%
group_by(CAD) %>%
count()
head(cad_prevent_input_hba1c_uacr)

In [ ]:
cad_prevent_input_hba1c = pheno_pgs %>%
select(eid,
       CAD,
       AGE_CAD,
       SEX,
       DIABETES_CAD,
       SBP_CAD,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_CAD,
       HDL_CAD,
       SMOKING_status_CAD,
       eGFR_CYS_C_CAD,
       BMI_CAD,
       HbA1c_CAD) %>%
rename('age' = 'AGE_CAD',
       'sex' = 'SEX',
       'sbp' = 'SBP_CAD',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_CAD',
       'hdl_c' = "HDL_CAD",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_CAD',
       'egfr' = 'eGFR_CYS_C_CAD',
       'dm' = 'DIABETES_CAD',
       'bmi' = 'BMI_CAD',
       'hba1c' = 'HbA1c_CAD') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       hba1c = ifelse(hba1c < 4.5 | hba1c > 15, NA, hba1c),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(cad_prevent_input_hba1c)
cad_prevent_input_hba1c %>%
group_by(CAD) %>%
count()
head(cad_prevent_input_hba1c)

In [ ]:
cad_prevent_input_uacr = pheno_pgs %>%
select(eid,
       CAD,
       AGE_CAD,
       SEX,
       DIABETES_CAD,
       SBP_CAD,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_CAD,
       HDL_CAD,
       SMOKING_status_CAD,
       eGFR_CYS_C_CAD,
       BMI_CAD,
       uACR_CAD) %>%
rename('age' = 'AGE_CAD',
       'sex' = 'SEX',
       'sbp' = 'SBP_CAD',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_CAD',
       'hdl_c' = "HDL_CAD",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_CAD',
       'egfr' = 'eGFR_CYS_C_CAD',
       'dm' = 'DIABETES_CAD',
       'bmi' = 'BMI_CAD',
       'uacr' = 'uACR_CAD') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       uacr = ifelse(uacr < 0.1 | uacr > 25000, NA, uacr),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()
nrow(cad_prevent_input_uacr)
cad_prevent_input_uacr %>%
group_by(CAD) %>%
count()
head(cad_prevent_input_uacr)

### remove overlapping samples

In [ ]:
cad_prevent_input_clean = cad_prevent_input %>%
filter(!eid %in% cad_prevent_input_uacr$eid) %>%
filter(!eid %in% cad_prevent_input_hba1c$eid) %>%
filter(!eid %in% cad_prevent_input_hba1c_uacr$eid)

nrow(cad_prevent_input_clean)
cad_prevent_input_clean %>%
group_by(CAD) %>%
count()

In [ ]:
cad_prevent_input_hba1c_clean = cad_prevent_input_hba1c %>%
filter(!eid %in% cad_prevent_input_hba1c_uacr$eid)

nrow(cad_prevent_input_hba1c_clean)
cad_prevent_input_hba1c_clean %>%
group_by(CAD) %>%
count()

In [ ]:
cad_prevent_input_uacr_clean = cad_prevent_input_uacr %>%
filter(!eid %in% cad_prevent_input_hba1c_uacr$eid)

nrow(cad_prevent_input_uacr_clean)
cad_prevent_input_uacr_clean %>%
group_by(CAD) %>%
count()

### check that no one overlaps

In [ ]:
nrow(cad_prevent_input_hba1c_uacr[cad_prevent_input_hba1c_uacr$eid %in% cad_prevent_input_clean$eid])
nrow(cad_prevent_input_hba1c_uacr[cad_prevent_input_hba1c_uacr$eid %in% cad_prevent_input_hba1c_clean$eid])
nrow(cad_prevent_input_hba1c_uacr[cad_prevent_input_hba1c_uacr$eid %in% cad_prevent_input_uacr_clean$eid])

In [ ]:
nrow(cad_prevent_input_hba1c_clean[cad_prevent_input_hba1c_clean$eid %in% cad_prevent_input_clean$eid])
nrow(cad_prevent_input_hba1c_clean[cad_prevent_input_hba1c_clean$eid %in% cad_prevent_input_uacr_clean$eid])

In [ ]:
nrow(cad_prevent_input_uacr_clean[cad_prevent_input_uacr_clean$eid %in% cad_prevent_input_clean$eid])
nrow(cad_prevent_input_uacr_clean[cad_prevent_input_uacr_clean$eid %in% cad_prevent_input_hba1c_clean$eid])

## AFIB

### subset and rename

In [ ]:
afib_prevent_input = pheno_pgs %>%
select(eid,
       AFIB,
       AGE_AFIB,
       SEX,
       DIABETES_AFIB,
       SBP_AFIB,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_AFIB,
       HDL_AFIB,
       SMOKING_status_AFIB,
       eGFR_CYS_C_AFIB,
       BMI_AFIB) %>%
rename('age' = 'AGE_AFIB',
       'sex' = 'SEX',
       'sbp' = 'SBP_AFIB',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_AFIB',
       'hdl_c' = "HDL_AFIB",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_AFIB',
       'egfr' = 'eGFR_CYS_C_AFIB',
       'dm' = 'DIABETES_AFIB',
       'bmi' = 'BMI_AFIB') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(afib_prevent_input)
afib_prevent_input %>%
group_by(AFIB) %>%
count()
head(afib_prevent_input)

In [ ]:
afib_prevent_input_hba1c_uacr = pheno_pgs %>%
select(eid,
       AFIB,
       AGE_AFIB,
       SEX,
       DIABETES_AFIB,
       SBP_AFIB,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_AFIB,
       HDL_AFIB,
       SMOKING_status_AFIB,
       eGFR_CYS_C_AFIB,
       BMI_AFIB,
       HbA1c_AFIB,
       uACR_AFIB) %>%
rename('age' = 'AGE_AFIB',
       'sex' = 'SEX',
       'sbp' = 'SBP_AFIB',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_AFIB',
       'hdl_c' = "HDL_AFIB",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_AFIB',
       'egfr' = 'eGFR_CYS_C_AFIB',
       'dm' = 'DIABETES_AFIB',
       'bmi' = 'BMI_AFIB',
       'hba1c' = 'HbA1c_AFIB',
       'uacr' = 'uACR_AFIB') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       uacr = ifelse(uacr < 0.1 | uacr > 25000, NA, uacr),
       hba1c = ifelse(hba1c < 4.5 | hba1c > 15, NA, hba1c),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(afib_prevent_input_hba1c_uacr)
afib_prevent_input_hba1c_uacr %>%
group_by(AFIB) %>%
count()
head(afib_prevent_input_hba1c_uacr)

In [ ]:
afib_prevent_input_hba1c = pheno_pgs %>%
select(eid,
       AFIB,
       AGE_AFIB,
       SEX,
       DIABETES_AFIB,
       SBP_AFIB,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_AFIB,
       HDL_AFIB,
       SMOKING_status_AFIB,
       eGFR_CYS_C_AFIB,
       BMI_AFIB,
       HbA1c_AFIB) %>%
rename('age' = 'AGE_AFIB',
       'sex' = 'SEX',
       'sbp' = 'SBP_AFIB',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_AFIB',
       'hdl_c' = "HDL_AFIB",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_AFIB',
       'egfr' = 'eGFR_CYS_C_AFIB',
       'dm' = 'DIABETES_AFIB',
       'bmi' = 'BMI_AFIB',
       'hba1c' = 'HbA1c_AFIB') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       hba1c = ifelse(hba1c < 4.5 | hba1c > 15, NA, hba1c),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(afib_prevent_input_hba1c)
afib_prevent_input_hba1c %>%
group_by(AFIB) %>%
count()
head(afib_prevent_input_hba1c)

In [ ]:
afib_prevent_input_uacr = pheno_pgs %>%
select(eid,
       AFIB,
       AGE_AFIB,
       SEX,
       DIABETES_AFIB,
       SBP_AFIB,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_AFIB,
       HDL_AFIB,
       SMOKING_status_AFIB,
       eGFR_CYS_C_AFIB,
       BMI_AFIB,
       uACR_AFIB) %>%
rename('age' = 'AGE_AFIB',
       'sex' = 'SEX',
       'sbp' = 'SBP_AFIB',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_AFIB',
       'hdl_c' = "HDL_AFIB",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_AFIB',
       'egfr' = 'eGFR_CYS_C_AFIB',
       'dm' = 'DIABETES_AFIB',
       'bmi' = 'BMI_AFIB',
       'uacr' = 'uACR_AFIB') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       uacr = ifelse(uacr < 0.1 | uacr > 25000, NA, uacr),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(afib_prevent_input_uacr)
afib_prevent_input_uacr %>%
group_by(AFIB) %>%
count()
head(afib_prevent_input_uacr)

### remove overlapping samples

In [ ]:
afib_prevent_input_clean = afib_prevent_input %>%
filter(!eid %in% afib_prevent_input_uacr$eid) %>%
filter(!eid %in% afib_prevent_input_hba1c$eid) %>%
filter(!eid %in% afib_prevent_input_hba1c_uacr$eid)

nrow(afib_prevent_input_clean)
afib_prevent_input_clean %>%
group_by(AFIB) %>%
count()

In [ ]:
afib_prevent_input_hba1c_clean = afib_prevent_input_hba1c %>%
filter(!eid %in% afib_prevent_input_hba1c_uacr$eid)

nrow(afib_prevent_input_hba1c_clean)
afib_prevent_input_hba1c_clean %>%
group_by(AFIB) %>%
count()

In [ ]:
afib_prevent_input_uacr_clean = afib_prevent_input_uacr %>%
filter(!eid %in% afib_prevent_input_hba1c_uacr$eid)

nrow(afib_prevent_input_uacr_clean)
afib_prevent_input_uacr_clean %>%
group_by(AFIB) %>%
count()

### check for overlapping samples

In [ ]:
nrow(afib_prevent_input_hba1c_uacr[afib_prevent_input_hba1c_uacr$eid %in% afib_prevent_input_clean$eid])
nrow(afib_prevent_input_hba1c_uacr[afib_prevent_input_hba1c_uacr$eid %in% afib_prevent_input_hba1c_clean$eid])
nrow(afib_prevent_input_hba1c_uacr[afib_prevent_input_hba1c_uacr$eid %in% afib_prevent_input_uacr_clean$eid])

In [ ]:
nrow(afib_prevent_input_hba1c_clean[afib_prevent_input_hba1c_clean$eid %in% afib_prevent_input_clean$eid])
nrow(afib_prevent_input_hba1c_clean[afib_prevent_input_hba1c_clean$eid %in% afib_prevent_input_uacr_clean$eid])

In [ ]:
nrow(afib_prevent_input_uacr_clean[afib_prevent_input_uacr_clean$eid %in% afib_prevent_input_clean$eid])
nrow(afib_prevent_input_uacr_clean[afib_prevent_input_uacr_clean$eid %in% afib_prevent_input_hba1c_clean$eid])

## HF

### subset and rename

In [ ]:
hf_prevent_input = pheno_pgs %>%
select(eid,
       HF,
       AGE_HF,
       SEX,
       DIABETES_HF,
       SBP_HF,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_HF,
       HDL_HF,
       SMOKING_status_HF,
       eGFR_CYS_C_HF,
       BMI_HF) %>%
rename('age' = 'AGE_HF',
       'sex' = 'SEX',
       'sbp' = 'SBP_HF',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_HF',
       'hdl_c' = "HDL_HF",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_HF',
       'egfr' = 'eGFR_CYS_C_HF',
       'dm' = 'DIABETES_HF',
       'bmi' = 'BMI_HF') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(hf_prevent_input)
hf_prevent_input %>%
group_by(HF) %>%
count()
head(hf_prevent_input)

In [ ]:
hf_prevent_input_hba1c_uacr = pheno_pgs %>%
select(eid,
       HF,
       AGE_HF,
       SEX,
       DIABETES_HF,
       SBP_HF,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_HF,
       HDL_HF,
       SMOKING_status_HF,
       eGFR_CYS_C_HF,
       BMI_HF,
       HbA1c_HF,
       uACR_HF) %>%
rename('age' = 'AGE_HF',
       'sex' = 'SEX',
       'sbp' = 'SBP_HF',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_HF',
       'hdl_c' = "HDL_HF",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_HF',
       'egfr' = 'eGFR_CYS_C_HF',
       'dm' = 'DIABETES_HF',
       'bmi' = 'BMI_HF',
       'hba1c' = 'HbA1c_HF',
       'uacr' = 'uACR_HF') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       uacr = ifelse(uacr < 0.1 | uacr > 25000, NA, uacr),
       hba1c = ifelse(hba1c < 4.5 | hba1c > 15, NA, hba1c),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(hf_prevent_input_hba1c_uacr)
hf_prevent_input_hba1c_uacr %>%
group_by(HF) %>%
count()
head(hf_prevent_input_hba1c_uacr)

In [ ]:
hf_prevent_input_hba1c = pheno_pgs %>%
select(eid,
       HF,
       AGE_HF,
       SEX,
       DIABETES_HF,
       SBP_HF,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_HF,
       HDL_HF,
       SMOKING_status_HF,
       eGFR_CYS_C_HF,
       BMI_HF,
       HbA1c_HF) %>%
rename('age' = 'AGE_HF',
       'sex' = 'SEX',
       'sbp' = 'SBP_HF',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_HF',
       'hdl_c' = "HDL_HF",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_HF',
       'egfr' = 'eGFR_CYS_C_HF',
       'dm' = 'DIABETES_HF',
       'bmi' = 'BMI_HF',
       'hba1c' = 'HbA1c_HF') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       hba1c = ifelse(hba1c < 4.5 | hba1c > 15, NA, hba1c),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(hf_prevent_input_hba1c)
hf_prevent_input_hba1c %>%
group_by(HF) %>%
count()
head(hf_prevent_input_hba1c)

In [ ]:
hf_prevent_input_uacr = pheno_pgs %>%
select(eid,
       HF,
       AGE_HF,
       SEX,
       DIABETES_HF,
       SBP_HF,
       MED_chol,
       MED_bp,
       TOTAL_CHOL_HF,
       HDL_HF,
       SMOKING_status_HF,
       eGFR_CYS_C_HF,
       BMI_HF,
       uACR_HF) %>%
rename('age' = 'AGE_HF',
       'sex' = 'SEX',
       'sbp' = 'SBP_HF',
       'bp_tx' = 'MED_bp',
       'total_c' = 'TOTAL_CHOL_HF',
       'hdl_c' = "HDL_HF",
       'statin' = 'MED_chol',
       'smoking' = 'SMOKING_status_HF',
       'egfr' = 'eGFR_CYS_C_HF',
       'dm' = 'DIABETES_HF',
       'bmi' = 'BMI_HF',
       'uacr' = 'uACR_HF') %>%
mutate(sex = case_when(sex == 1 ~ 'male', sex == 2 ~ 'female'),
       age = ifelse(age < 30 | age > 79, NA, age),
       hdl_c = ifelse(hdl_c < 20 | hdl_c > 100, NA, hdl_c),
       total_c = ifelse(total_c < 130 | total_c > 320, NA, total_c),
       bmi = ifelse(bmi < 18.5 | bmi > 39, NA, bmi),
       sbp = ifelse(sbp < 90 | sbp > 180, NA, sbp),
       uacr = ifelse(uacr < 0.1 | uacr > 25000, NA, uacr),
       smoking = ifelse(smoking == 1, 0, smoking),
       smoking = ifelse(smoking == 2, 1, smoking),
       egfr = ifelse(egfr < 15 | egfr > 140, NA, egfr)) %>%
na.omit()

nrow(hf_prevent_input_uacr)
hf_prevent_input_uacr %>%
group_by(HF) %>%
count()
head(hf_prevent_input_uacr)

### remove overlapping individuals

In [ ]:
hf_prevent_input_clean = hf_prevent_input %>%
filter(!eid %in% hf_prevent_input_uacr$eid) %>%
filter(!eid %in% hf_prevent_input_hba1c$eid) %>%
filter(!eid %in% hf_prevent_input_hba1c_uacr$eid)

nrow(hf_prevent_input_clean)
hf_prevent_input_clean %>%
group_by(HF) %>%
count()

In [ ]:
hf_prevent_input_hba1c_clean = hf_prevent_input_hba1c %>%
filter(!eid %in% hf_prevent_input_hba1c_uacr$eid)

nrow(hf_prevent_input_hba1c_clean)
hf_prevent_input_hba1c_clean %>%
group_by(HF) %>%
count()

In [ ]:
hf_prevent_input_uacr_clean = hf_prevent_input_uacr %>%
filter(!eid %in% hf_prevent_input_hba1c_uacr$eid)

nrow(hf_prevent_input_uacr_clean)
hf_prevent_input_uacr_clean %>%
group_by(HF) %>%
count()

### check that no one overlaps

In [ ]:
nrow(hf_prevent_input_hba1c_uacr[hf_prevent_input_hba1c_uacr$eid %in% hf_prevent_input_clean$eid])
nrow(hf_prevent_input_hba1c_uacr[hf_prevent_input_hba1c_uacr$eid %in% hf_prevent_input_hba1c_clean$eid])
nrow(hf_prevent_input_hba1c_uacr[hf_prevent_input_hba1c_uacr$eid %in% hf_prevent_input_uacr_clean$eid])

In [ ]:
nrow(hf_prevent_input_hba1c_clean[hf_prevent_input_hba1c_clean$eid %in% hf_prevent_input_clean$eid])
nrow(hf_prevent_input_hba1c_clean[hf_prevent_input_hba1c_clean$eid %in% hf_prevent_input_uacr_clean$eid])

In [ ]:
nrow(hf_prevent_input_uacr_clean[hf_prevent_input_uacr_clean$eid %in% hf_prevent_input_clean$eid])
nrow(hf_prevent_input_uacr_clean[hf_prevent_input_uacr_clean$eid %in% hf_prevent_input_hba1c_clean$eid])

# compute risk

## CAD

In [ ]:
cad_crs = est_risk(use_dat = cad_prevent_input_clean)
unique(cad_crs$input_problems)
dim(cad_crs)
length(unique(cad_crs$person_id))
head(cad_crs)

In [ ]:
cad_crs_hba1c = est_risk(use_dat = cad_prevent_input_hba1c_clean)
unique(cad_crs_hba1c$input_problems)
dim(cad_crs_hba1c)
length(unique(cad_crs_hba1c$person_id))
head(cad_crs_hba1c)

In [ ]:
cad_crs_uacr = est_risk(use_dat = cad_prevent_input_uacr_clean)
unique(cad_crs_uacr$input_problems)
dim(cad_crs_uacr)
length(unique(cad_crs_uacr$person_id))
head(cad_crs_uacr)

In [ ]:
cad_crs_hba1c_uacr = est_risk(use_dat = cad_prevent_input_hba1c_uacr)
unique(cad_crs_hba1c_uacr$input_problems)
dim(cad_crs_hba1c_uacr)
length(unique(cad_crs_hba1c_uacr$person_id))
head(cad_crs_hba1c_uacr)

## AFIB

In [ ]:
afib_crs = est_risk(use_dat = afib_prevent_input_clean)
unique(afib_crs$input_problems)
dim(afib_crs)
length(unique(afib_crs$person_id))
head(afib_crs)

In [ ]:
afib_crs_hba1c = est_risk(use_dat = afib_prevent_input_hba1c_clean)
unique(afib_crs_hba1c$input_problems)
dim(afib_crs_hba1c)
length(unique(afib_crs_hba1c$person_id))
head(afib_crs_hba1c)

In [ ]:
afib_crs_uacr = est_risk(use_dat = afib_prevent_input_uacr_clean)
unique(afib_crs_uacr$input_problems)
dim(afib_crs_uacr)
length(unique(afib_crs_uacr$person_id))
head(afib_crs_uacr)

In [ ]:
afib_crs_hba1c_uacr = est_risk(use_dat = afib_prevent_input_hba1c_uacr)
unique(afib_crs_hba1c_uacr$input_problems)
dim(afib_crs_hba1c_uacr)
length(unique(afib_crs_hba1c_uacr$person_id))
head(afib_crs_hba1c_uacr)

## HF

In [ ]:
hf_crs = est_risk(use_dat = hf_prevent_input_clean)
unique(hf_crs$input_problems)
dim(hf_crs)
length(unique(hf_crs$person_id))
head(hf_crs)

In [ ]:
hf_crs_hba1c = est_risk(use_dat = hf_prevent_input_hba1c_clean)
unique(hf_crs_hba1c$input_problems)
dim(hf_crs_hba1c)
length(unique(hf_crs_hba1c$person_id))
head(hf_crs_hba1c)

In [ ]:
hf_crs_uacr = est_risk(use_dat = hf_prevent_input_uacr_clean)
unique(hf_crs_uacr$input_problems)
dim(hf_crs_uacr)
length(unique(hf_crs_uacr$person_id))
head(hf_crs_uacr)

In [ ]:
hf_crs_hba1c_uacr = est_risk(use_dat = hf_prevent_input_hba1c_uacr)
unique(hf_crs_hba1c_uacr$input_problems)
dim(hf_crs_hba1c_uacr)
length(unique(hf_crs_hba1c_uacr$person_id))
head(hf_crs_hba1c_uacr)

# subset CRS dfs

## cad

In [ ]:
cad_crs_sub = cad_crs %>%
select(eid, CAD, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(cad_crs_sub)

In [ ]:
cad_crs_hba1c_sub = cad_crs_hba1c %>%
select(eid, CAD, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(cad_crs_hba1c_sub)

In [ ]:
cad_crs_uacr_sub = cad_crs_uacr %>%
select(eid, CAD, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(cad_crs_uacr_sub)

In [ ]:
cad_crs_hba1c_uacr_sub = cad_crs_hba1c_uacr %>%
select(eid, CAD, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(cad_crs_hba1c_uacr_sub)

## afib

In [ ]:
afib_crs_sub = afib_crs %>%
select(eid, AFIB, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(afib_crs_sub)

In [ ]:
afib_crs_hba1c_sub = afib_crs_hba1c %>%
select(eid, AFIB, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(afib_crs_hba1c_sub)

In [ ]:
afib_crs_uacr_sub = afib_crs_uacr %>%
select(eid, AFIB, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(afib_crs_uacr_sub)

In [ ]:
afib_crs_hba1c_uacr_sub = afib_crs_hba1c_uacr %>%
select(eid, AFIB, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(afib_crs_hba1c_uacr_sub)

## HF

In [ ]:
hf_crs_sub = hf_crs %>%
select(eid, HF, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(hf_crs_sub)

In [ ]:
hf_crs_hba1c_sub = hf_crs_hba1c %>%
select(eid, HF, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(hf_crs_hba1c_sub)

In [ ]:
hf_crs_hba1c_uacr_sub = hf_crs_hba1c_uacr %>%
select(eid, HF, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(hf_crs_hba1c_uacr_sub)

In [ ]:
hf_crs_uacr_sub = hf_crs_uacr %>%
select(eid, HF, age, total_cvd, ascvd, heart_failure, chd, stroke, model, over_years)
head(hf_crs_uacr_sub)

# concat

In [ ]:
cad_crs_all = rbind(cad_crs_sub, cad_crs_hba1c_sub, cad_crs_uacr_sub, cad_crs_hba1c_uacr_sub)
nrow(cad_crs_all)
head(cad_crs_all)

In [ ]:
afib_crs_all = rbind(afib_crs_sub, afib_crs_hba1c_sub, afib_crs_uacr_sub, afib_crs_hba1c_uacr_sub)
nrow(afib_crs_all)
head(afib_crs_all)

In [ ]:
hf_crs_all = rbind(hf_crs_sub, hf_crs_hba1c_sub, hf_crs_uacr_sub, hf_crs_hba1c_uacr_sub)
nrow(hf_crs_all)
head(hf_crs_all)

# split into 10 and 30 year risk dfs

## cad

In [ ]:
cad_crs_10 = cad_crs_all %>%
filter(over_years == 10)
nrow(cad_crs_10)
length(unique(cad_crs_10$person_id))

In [ ]:
cad_crs_30 = cad_crs_all %>%
filter(over_years == 30)
nrow(cad_crs_30)
length(unique(cad_crs_30$person_id))

## afib

In [ ]:
afib_crs_10 = afib_crs_all %>%
filter(over_years == 10)
nrow(afib_crs_10)
length(unique(afib_crs_10$person_id))

In [ ]:
afib_crs_30 = afib_crs_all %>%
filter(over_years == 30)
nrow(afib_crs_30)
length(unique(afib_crs_30$person_id))

## hf

In [ ]:
hf_crs_10 = hf_crs_all %>%
filter(over_years == 10)
nrow(hf_crs_10)
length(unique(hf_crs_10$person_id))

In [ ]:
hf_crs_30 = hf_crs_all %>%
filter(over_years == 30)
nrow(hf_crs_30)
length(unique(hf_crs_30$person_id))

# export

## cad

In [ ]:
write.table(cad_crs_10,
            'UKBB.CAD.PREVENT_CRS.10_year_risk.txt',
            sep = '\t',
            col.names = T,
            row.names = F,
            quote = F)

In [ ]:
write.table(cad_crs_30,
            'UKBB.CAD.PREVENT_CRS.30_year_risk.txt',
            sep = '\t',
            col.names = T,
            row.names = F,
            quote = F)

## afib

In [ ]:
write.table(afib_crs_10,
            'UKBB.AFIB.PREVENT_CRS.10_year_risk.txt',
            sep = '\t',
            col.names = T,
            row.names = F,
            quote = F)

In [ ]:
write.table(afib_crs_30,
            'UKBB.AFIB.PREVENT_CRS.30_year_risk.txt',
            sep = '\t',
            col.names = T,
            row.names = F,
            quote = F)

## HF

In [ ]:
write.table(hf_crs_10,
            'UKBB.HF.PREVENT_CRS.10_year_risk.txt',
            sep = '\t',
            col.names = T,
            row.names = F,
            quote = F)

In [ ]:
write.table(hf_crs_30,
            'UKBB.HF.PREVENT_CRS.30_year_risk.txt',
            sep = '\t',
            col.names = T,
            row.names = F,
            quote = F)